# US Traffic Accident Risk EDA

This notebook analyzes the US-only datasets used by the project pipeline.

Project data contract:

- Records before `2020` are used for offline pretraining.
- Records from `2020` onward simulate realtime Kafka replay and later retraining.
- `true_severity` is the normalized ground-truth label copied from the US Accidents `Severity` column.
- Feature analysis focuses on the processed feature files under `data/process/`. If a replay file is not present there, the notebook falls back to `data/split/` for raw replay inspection.


In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 100)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

PROCESS_DIR = PROJECT_ROOT / "data" / "process"
SPLIT_DIR = PROJECT_ROOT / "data" / "split"
RAW_DIR = PROJECT_ROOT / "data" / "raw"

OFFLINE_FEATURE_PATH = PROCESS_DIR / "us_train_offline_before_2020.csv"
REPLAY_FEATURE_PATH = PROCESS_DIR / "us_pipeline_from_2020.csv"
REPLAY_RAW_FALLBACK_PATH = SPLIT_DIR / "us_pipeline_from_2020.csv"
RAW_US_PATH = RAW_DIR / "US.csv"

SPLIT_YEAR = 2020
SAMPLE_ROWS = 500_000

print(f"Project root: {PROJECT_ROOT}")
print(f"Processed data directory: {PROCESS_DIR}")
print(f"Offline feature path: {OFFLINE_FEATURE_PATH}")
print(f"Replay feature path: {REPLAY_FEATURE_PATH}")


## 1. Load Processed Data

Large CSV files are sampled to keep the notebook responsive. The sample is enough for distribution, imbalance, and correlation analysis while preserving the full file path contract used by the pipeline.


In [ ]:
def load_csv_sample(path: Path, label: str, sample_rows: int = SAMPLE_ROWS) -> pd.DataFrame:
    """Load a CSV file with a deterministic cap for local EDA responsiveness."""
    if not path.exists():
        print(f"{label}: missing file: {path}")
        return pd.DataFrame()

    print(f"Loading {label}: {path}")
    df = pd.read_csv(path, nrows=sample_rows, low_memory=False)
    print(f"{label}: loaded {len(df):,} rows and {len(df.columns):,} columns")
    return df


offline_df = load_csv_sample(OFFLINE_FEATURE_PATH, "offline_before_2020")

if REPLAY_FEATURE_PATH.exists():
    replay_df = load_csv_sample(REPLAY_FEATURE_PATH, "replay_from_2020_processed")
elif REPLAY_RAW_FALLBACK_PATH.exists():
    replay_df = load_csv_sample(REPLAY_RAW_FALLBACK_PATH, "replay_from_2020_raw_fallback")
else:
    replay_df = pd.DataFrame()
    print("No replay file found in data/process or data/split.")

display(offline_df.head())
display(replay_df.head())


In [ ]:
def standardize_for_eda(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize raw or processed US columns into the project feature names for EDA."""
    if df.empty:
        return df.copy()

    result = df.copy()
    if "true_severity" not in result.columns and "Severity" in result.columns:
        result["true_severity"] = pd.to_numeric(result["Severity"], errors="coerce")
    if "event_time" not in result.columns and "Start_Time" in result.columns:
        result["event_time"] = pd.to_datetime(result["Start_Time"], errors="coerce")
    elif "event_time" in result.columns:
        result["event_time"] = pd.to_datetime(result["event_time"], errors="coerce")

    if "event_year" not in result.columns and "event_time" in result.columns:
        result["event_year"] = result["event_time"].dt.year
    if "hour" not in result.columns and "event_time" in result.columns:
        result["hour"] = result["event_time"].dt.hour
    if "day_of_week" not in result.columns and "event_time" in result.columns:
        result["day_of_week"] = (result["event_time"].dt.dayofweek + 2).where(result["event_time"].dt.dayofweek < 6, 1)

    rename_map = {
        "Start_Lat": "lat",
        "Start_Lng": "lon",
        "Temperature(F)": "temperature_f",
        "Humidity(%)": "humidity",
        "Wind_Speed(mph)": "wind_speed_mph",
        "Visibility(mi)": "visibility_mi",
    }
    for source, target in rename_map.items():
        if target not in result.columns and source in result.columns:
            result[target] = pd.to_numeric(result[source], errors="coerce")

    return result


offline_eda = standardize_for_eda(offline_df)
replay_eda = standardize_for_eda(replay_df)
combined_eda = pd.concat(
    [offline_eda.assign(dataset="offline_before_2020"), replay_eda.assign(dataset="replay_from_2020")],
    ignore_index=True,
)

print("Combined EDA shape:", combined_eda.shape)
display(combined_eda[[c for c in ["dataset", "event_year", "event_time", "true_severity", "lat", "lon", "hour"] if c in combined_eda.columns]].head())


## 2. Data Quality and Split Validation

This section checks missing values, duplicate event identifiers, geographic bounds, and whether the 2020 split matches the intended offline/realtime contract.


In [ ]:
def quality_report(df: pd.DataFrame, label: str) -> pd.DataFrame:
    """Build a compact data quality summary for one dataset."""
    if df.empty:
        return pd.DataFrame([{"dataset": label, "rows": 0}])

    duplicate_events = int(df["event_id"].duplicated().sum()) if "event_id" in df.columns else None
    invalid_severity = int((~df["true_severity"].between(1, 4)).sum()) if "true_severity" in df.columns else None
    invalid_lat = int((~df["lat"].between(-90, 90)).sum()) if "lat" in df.columns else None
    invalid_lon = int((~df["lon"].between(-180, 180)).sum()) if "lon" in df.columns else None
    return pd.DataFrame([
        {
            "dataset": label,
            "rows": len(df),
            "columns": len(df.columns),
            "duplicate_event_ids": duplicate_events,
            "invalid_true_severity": invalid_severity,
            "invalid_lat": invalid_lat,
            "invalid_lon": invalid_lon,
            "min_year": int(df["event_year"].min()) if "event_year" in df.columns and df["event_year"].notna().any() else None,
            "max_year": int(df["event_year"].max()) if "event_year" in df.columns and df["event_year"].notna().any() else None,
        }
    ])


display(pd.concat([quality_report(offline_eda, "offline_before_2020"), quality_report(replay_eda, "replay_from_2020")]))

missing = (combined_eda.isna().mean().sort_values(ascending=False) * 100).round(2).to_frame("missing_pct")
display(missing.head(30))


## 3. Class Imbalance

`true_severity` is expected to be imbalanced in the US Accidents dataset. The pie chart and bar chart make the imbalance visible for model training decisions such as class balancing in H2O AutoML.


In [ ]:
severity_counts = combined_eda["true_severity"].dropna().astype(int).value_counts().sort_index()
display(severity_counts.to_frame("rows"))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].pie(severity_counts.values, labels=severity_counts.index, autopct="%1.1f%%", startangle=90)
axes[0].set_title("true_severity class share")

sns.barplot(x=severity_counts.index.astype(str), y=severity_counts.values, ax=axes[1], color="#4c78a8")
axes[1].set_title("true_severity class counts")
axes[1].set_xlabel("true_severity")
axes[1].set_ylabel("rows")
plt.tight_layout()
plt.show()


## 4. Numeric Distributions

Histograms show whether model inputs have extreme skew, missing-value defaults, or impossible ranges that should be handled before training or inference.


In [ ]:
numeric_features = [
    "lat", "lon", "hour", "day_of_week", "weather_code", "temperature_f",
    "humidity", "wind_speed_mph", "visibility_mi", "road_type_code",
    "is_junction", "has_traffic_signal", "is_crossing", "is_roundabout",
    "is_stop", "is_station", "is_railway", "is_night",
]
available_numeric = [column for column in numeric_features if column in combined_eda.columns]

plot_df = combined_eda[available_numeric].apply(pd.to_numeric, errors="coerce")
axes = plot_df.hist(figsize=(18, 14), bins=40, color="#4c78a8", edgecolor="white")
for axis in axes.flatten():
    axis.set_title(axis.get_title(), fontsize=10)
plt.tight_layout()
plt.show()

display(plot_df.describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T.round(3))


## 5. Correlation Heatmap

The heatmap checks linear relationships between model features and `true_severity`. It is not a full feature-importance method, but it quickly surfaces redundant or suspiciously coupled columns.


In [ ]:
corr_columns = ["true_severity"] + available_numeric
corr_columns = list(dict.fromkeys([column for column in corr_columns if column in combined_eda.columns]))
corr_df = combined_eda[corr_columns].apply(pd.to_numeric, errors="coerce").dropna(axis=1, how="all")
corr = corr_df.corr(numeric_only=True)

plt.figure(figsize=(14, 10))
sns.heatmap(corr, cmap="vlag", center=0, square=False, linewidths=0.3)
plt.title("Feature correlation heatmap")
plt.tight_layout()
plt.show()

if "true_severity" in corr.columns:
    display(corr["true_severity"].drop("true_severity", errors="ignore").sort_values(key=lambda s: s.abs(), ascending=False).to_frame("corr_with_true_severity"))


## 6. Time, Weather, and Road Context Patterns

These charts connect risk labels to interpretable operational features that are useful for dashboard storytelling and scenario simulation.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

if "event_year" in combined_eda.columns:
    yearly = combined_eda.groupby(["event_year", "dataset"]).size().reset_index(name="rows")
    sns.lineplot(data=yearly, x="event_year", y="rows", hue="dataset", marker="o", ax=axes[0, 0])
    axes[0, 0].axvline(SPLIT_YEAR, color="red", linestyle="--", linewidth=1)
    axes[0, 0].set_title("Rows by year and pipeline split")

if "hour" in combined_eda.columns:
    hourly = combined_eda.groupby("hour").agg(rows=("hour", "size"), avg_severity=("true_severity", "mean")).reset_index()
    sns.barplot(data=hourly, x="hour", y="rows", color="#72b7b2", ax=axes[0, 1])
    axes[0, 1].set_title("Accident volume by hour")

if "weather_code" in combined_eda.columns:
    weather = combined_eda.groupby("weather_code").agg(rows=("weather_code", "size"), avg_severity=("true_severity", "mean")).reset_index()
    sns.barplot(data=weather, x="weather_code", y="avg_severity", color="#f58518", ax=axes[1, 0])
    axes[1, 0].set_title("Average severity by weather_code")

if "road_type_code" in combined_eda.columns:
    road = combined_eda.groupby("road_type_code").agg(rows=("road_type_code", "size"), avg_severity=("true_severity", "mean")).reset_index()
    sns.barplot(data=road, x="road_type_code", y="avg_severity", color="#54a24b", ax=axes[1, 1])
    axes[1, 1].set_title("Average severity by road_type_code")

plt.tight_layout()
plt.show()


## 7. Spatial Overview

A sampled scatter plot checks whether coordinates are within the expected US bounding box and shows rough geographic concentration without requiring a map library.


In [ ]:
if {"lat", "lon", "true_severity"}.issubset(combined_eda.columns):
    spatial_df = combined_eda[["lat", "lon", "true_severity", "dataset"]].dropna().sample(
        n=min(50_000, len(combined_eda.dropna(subset=["lat", "lon", "true_severity"]))),
        random_state=42,
    )
    plt.figure(figsize=(12, 7))
    sns.scatterplot(
        data=spatial_df,
        x="lon",
        y="lat",
        hue="true_severity",
        palette="viridis",
        s=8,
        alpha=0.45,
        linewidth=0,
    )
    plt.title("Sampled US accident locations by true_severity")
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.tight_layout()
    plt.show()


## 8. EDA Findings Template

Use the generated tables and charts to fill this section after running the notebook on the full processed files:

- Severity imbalance: class `2` is usually dominant; H2O class balancing should remain enabled.
- Time split: offline rows must be strictly before 2020; replay rows should start at 2020.
- Feature health: latitude, longitude, time, weather, and road-context features should have low missing rates after feature engineering.
- Dashboard relevance: `hour`, `weather_code`, `road_type_code`, and spatial clusters are directly useful for Live Map, Hotspots, and Scenario Simulator pages.
